# EMT 3-phase Piecewise-Linear Inductor

This notebook validates the EMT 3-phase `PiecewiseLinearInductor` against its defined piecewise-linear flux-current characteristic.

Two cases are simulated:

1. **without steady-state initialization**
2. **with steady-state initialization**

The notebook checks that:
- the operating point passes through several characteristic segments
- the simulated current stays on the defined characteristic, allowing a single-sample mismatch at segment changes
- invalid parameter sets raise exceptions
- the steady-state initialized case has no DC offset

In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import dpsimpy

emt = dpsimpy.emt
ph3 = emt.ph3

Simulation = dpsimpy.Simulation
SystemTopology = dpsimpy.SystemTopology
Logger = dpsimpy.Logger
Domain = dpsimpy.Domain
PhaseType = dpsimpy.PhaseType

## Parameters

In [14]:
time_step = 1e-4
final_time = 0.2

source_voltage_peak = 100.0
source_frequency = 50.0
source_phase_deg = -90.0
source_resistance = 0.2
base_current = 1.0

peak1ph_to_rms3ph = np.sqrt(3.0 / 2.0)

base_flux = np.sqrt(2.0) * source_voltage_peak / (2.0 * np.pi * source_frequency)

flux_in_per_unit = np.array([0.0, 0.5, 0.9, 1.0], dtype=float)
current_in_per_unit = np.array([0.0, 0.01, 0.1, 10.0], dtype=float)

flux_breakpoints = flux_in_per_unit * base_flux
current_breakpoints = current_in_per_unit * base_current

## Helpers

In [15]:
def abc(phasor):
    return np.array(
        [
            [phasor],
            [phasor * np.exp(-1j * 2 * np.pi / 3)],
            [phasor * np.exp(+1j * 2 * np.pi / 3)],
        ],
        dtype=np.complex128,
    )


def source_voltage_rms_line_to_line():
    # DPsim source uses cosine convention.
    return (
        peak1ph_to_rms3ph
        * source_voltage_peak
        * np.exp(1j * np.deg2rad(source_phase_deg))
    )


def first_segment_inductance():
    delta_flux = flux_breakpoints[1] - flux_breakpoints[0]
    delta_current = current_breakpoints[1] - current_breakpoints[0]
    return delta_flux / delta_current


def set_initial_node_voltages(n1, n2):
    v_source = source_voltage_rms_line_to_line()
    z_inductor = 1j * 2.0 * np.pi * source_frequency * first_segment_inductance()
    z_total = source_resistance + z_inductor

    i_steady_state = v_source / z_total
    v_inductor = i_steady_state * z_inductor

    n1.set_initial_voltage(v_source)
    n2.set_initial_voltage(v_inductor)


def segment_index(flux):
    abs_flux = np.abs(np.asarray(flux))
    idx = np.searchsorted(flux_breakpoints[1:], abs_flux, side="left")
    return np.clip(idx, 0, len(flux_breakpoints) - 2)


def characteristic_current(flux):
    flux = np.asarray(flux)
    idx = segment_index(flux)

    slopes = (current_breakpoints[1:] - current_breakpoints[:-1]) / (
        flux_breakpoints[1:] - flux_breakpoints[:-1]
    )
    offsets = current_breakpoints[:-1] - slopes * flux_breakpoints[:-1]

    sign = np.where(flux < 0.0, -1.0, 1.0)
    slope = slopes[idx]
    offset = sign * offsets[idx]

    return slope * flux + offset


def sig(df, name):
    cols = [c for c in df.columns if c == name or c.startswith(name + "_")]
    return df[cols].to_numpy()

## Run cases

In [16]:
def run_case(sim_name, start_in_steady_state):
    n1 = emt.SimNode("n1", PhaseType.ABC)
    n2 = emt.SimNode("n2", PhaseType.ABC)

    if start_in_steady_state:
        set_initial_node_voltages(n1, n2)

    vs = ph3.VoltageSource("VS")
    vs.set_parameters(abc(source_voltage_rms_line_to_line()), source_frequency)

    r_source = ph3.Resistor("RSource")
    r_source.set_parameters(source_resistance * np.eye(3))

    l_piecewise = ph3.PiecewiseLinearInductor("LPiecewiseLinear")
    l_piecewise.set_parameters(
        flux_breakpoints.tolist(),
        current_breakpoints.tolist(),
    )

    vs.connect([emt.SimNode.gnd, n1])
    r_source.connect([n1, n2])
    l_piecewise.connect([n2, emt.SimNode.gnd])

    system = SystemTopology(
        source_frequency,
        [n1, n2],
        [vs, r_source, l_piecewise],
    )

    Logger.set_log_dir("logs/" + sim_name)
    logger = Logger(sim_name)
    logger.log_attribute("voltage", n2.attr("v"))
    logger.log_attribute("current", l_piecewise.attr("i_intf"))
    logger.log_attribute("flux", l_piecewise.attr("x"))

    sim = Simulation(sim_name)
    sim.set_system(system)
    sim.add_logger(logger)
    sim.set_domain(Domain.EMT)
    sim.do_system_matrix_recomputation(True)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.run()

In [ ]:
run_case("EMT_Ph3_PiecewiseLinearInductor", start_in_steady_state=False)
run_case("EMT_Ph3_PiecewiseLinearInductor_SteadyStateInit", start_in_steady_state=True)

## Load CSV logs

In [18]:
no_init = pd.read_csv(
    Path("logs/EMT_Ph3_PiecewiseLinearInductor/EMT_Ph3_PiecewiseLinearInductor.csv"),
    skipinitialspace=True,
)

steady_init = pd.read_csv(
    Path(
        "logs/EMT_Ph3_PiecewiseLinearInductor_SteadyStateInit/EMT_Ph3_PiecewiseLinearInductor_SteadyStateInit.csv"
    ),
    skipinitialspace=True,
)

t_no_init = no_init.iloc[:, 0].to_numpy()
t_steady_init = steady_init.iloc[:, 0].to_numpy()

np.testing.assert_allclose(t_no_init, t_steady_init)

## Parameter exception checks

In [ ]:
def expect_parameter_exception(flux_bp, current_bp):
    comp = ph3.PiecewiseLinearInductor("L")
    try:
        comp.set_parameters(flux_bp, current_bp)
    except Exception:
        return
    raise AssertionError("Expected set_parameters() to raise an exception.")


expect_parameter_exception([0.0], [0.0])
expect_parameter_exception([0.0, 1.0], [0.0])
expect_parameter_exception([0.1, 1.0], [0.0, 1.0])
expect_parameter_exception([0.0, 1.0, 0.5], [0.0, 1.0, 2.0])
expect_parameter_exception([0.0, 1.0, 2.0], [0.0, 2.0, 1.0])

print("Parameter exception checks passed.")

## Characteristic adherence check

In [ ]:
def assert_characteristic_adherence(df, label, require_multiple_segments=False):
    current = sig(df, "current")
    flux = sig(df, "flux")

    current_tolerance = 2e-4

    for phase in range(3):
        phase_flux = flux[:, phase]
        phase_current = current[:, phase]

        seg = segment_index(phase_flux)
        expected = characteristic_current(phase_flux)

        # Allow exactly one sample mismatch when the active segment changes.
        allowed = np.zeros_like(seg, dtype=bool)
        allowed[1:] = seg[1:] != seg[:-1]

        error = np.abs(phase_current - expected)
        max_error = error[~allowed].max(initial=0.0)

        assert np.all(error[~allowed] <= current_tolerance), (
            f"{label}: phase {phase} violates characteristic adherence away "
            f"from segment changes (max error {max_error:.3e})."
        )

        if require_multiple_segments:
            assert (
                len(np.unique(seg)) >= 3
            ), f"{label}: phase {phase} does not pass through several characteristic segments."


assert_characteristic_adherence(
    no_init,
    "without steady-state initialization",
    require_multiple_segments=True,
)
assert_characteristic_adherence(
    steady_init,
    "with steady-state initialization",
    require_multiple_segments=False,
)

print("Characteristic adherence checks passed.")

## Steady-state initialization check

In [ ]:
def assert_no_dc_offset(df, label):
    time = df.iloc[:, 0].to_numpy()
    current = sig(df, "current")

    period = 1.0 / source_frequency
    first_cycle = time < period

    cycle_mean = current[first_cycle].mean(axis=0)

    np.testing.assert_allclose(
        cycle_mean,
        np.zeros(3),
        atol=1e-4,
        rtol=0.0,
        err_msg=f"{label}: first-cycle mean current is not close to zero.",
    )


assert_no_dc_offset(steady_init, "steady-state initialized case")

print("Steady-state initialization check passed.")

## Flux-current consistency check

In [ ]:
def assert_flux_current_match(df, label):
    current = sig(df, "current")
    flux = sig(df, "flux")
    expected = characteristic_current(flux)

    seg = segment_index(flux)
    allowed = np.zeros_like(seg, dtype=bool)
    allowed[1:, :] = seg[1:, :] != seg[:-1, :]

    mask = ~allowed
    np.testing.assert_allclose(
        current[mask],
        expected[mask],
        atol=2e-4,
        rtol=0,
        err_msg=f"{label}: current does not match characteristic current.",
    )


assert_flux_current_match(no_init, "without steady-state initialization")
assert_flux_current_match(steady_init, "with steady-state initialization")

print("Flux-current consistency checks passed.")

## Plot phase currents

In [ ]:
plt.figure(figsize=(10, 5))

for phase, name in enumerate(["A", "B", "C"]):
    plt.plot(
        t_no_init,
        sig(no_init, "current")[:, phase],
        label=f"phase {name}",
    )

plt.xlabel("time [s]")
plt.ylabel("current [A]")
plt.title("Piecewise-linear inductor current without steady-state initialization")
plt.grid(True)
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))

for phase, name in enumerate(["A", "B", "C"]):
    plt.plot(
        t_steady_init,
        sig(steady_init, "current")[:, phase],
        label=f"phase {name}",
    )

plt.xlabel("time [s]")
plt.ylabel("current [A]")
plt.title("Piecewise-linear inductor current with steady-state initialization")
plt.grid(True)
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

## Plot current-flux characteristic

In [ ]:
phi_plot = np.linspace(-flux_breakpoints[-1], flux_breakpoints[-1], 1000)
i_plot = characteristic_current(phi_plot)

phase = 0  # phase A

plt.figure(figsize=(10, 5))

plt.plot(
    sig(no_init, "current")[:, phase],
    sig(no_init, "flux")[:, phase],
    alpha=0.8,
    label="phase A",
)

plt.plot(
    i_plot,
    phi_plot,
    color="black",
    linewidth=1.0,
    zorder=10,
    label="defined characteristic",
)

plt.xlabel("current [A]")
plt.ylabel("flux [Wb]")
plt.title("Flux-current characteristic without steady-state initialization")
plt.grid(True)
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))

plt.plot(
    sig(steady_init, "current")[:, phase],
    sig(steady_init, "flux")[:, phase],
    alpha=0.8,
    label="phase A",
)

plt.plot(
    i_plot,
    phi_plot,
    color="black",
    linewidth=1.0,
    zorder=10,
    label="defined characteristic",
)

plt.xlabel("current [A]")
plt.ylabel("flux [Wb]")
plt.title("Flux-current characteristic with steady-state initialization")
plt.grid(True)
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()